```
# Program: DuET v1.1.0
# Author: Sungho Lee, Jae-Won Lee
# Affiliation: MOGAM Institute for Biomedical Research
# Contact: https://github.com/mogam-ai/DuET/issues
# Citation: TBD
```

# DuET 5'UTR Motif Analysis

**Analysis-only notebook.** Parses raw outputs from the motif tools (run separately via shell scripts) and compares/visualizes them. Does **not** run the tools.

Inputs (produced by `run_xstreme.sh`, `run_modisco.sh`):
- `results/xstreme/<cond>/xstreme.tsv` - XSTREME discovered+enriched motifs
- `results/modisco/modisco_<tag>.h5` + `report_<tag>/` - TF-MoDISco motifs

Conditions: TE cut (te10/te20) x window (full_lenmatch / 5p{50,100} / 3p{50,100}).
5p=cap-proximal, 3p=start-proximal. high TE = primary, low TE = control.

**FINALIZED scope:** te10 cutoff + 100nt window only (5'p100 + 3'p100, hi + lo). RBP DB and TF-MoDISco sources are global (condition-independent) and kept in full.


In [ ]:
import os, glob, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# cwd-robust ROOT: works whether run from the DuET project root or from
# inside motif_search/ (or anywhere with motif_search reachable).
_cwd = Path.cwd()
if (_cwd / 'motif_search' / 'results').exists() or (_cwd / 'motif_search' / 'data').exists():
    ROOT = _cwd / 'motif_search'          # run from DuET root
elif _cwd.name == 'motif_search':
    ROOT = _cwd                            # run from inside motif_search/
elif (_cwd / 'data').exists() and (_cwd / 'results').exists():
    ROOT = _cwd                            # already at motif_search
else:
    ROOT = Path('motif_search')            # fallback (relative)
print('ROOT =', ROOT.resolve())
XS_DIR = ROOT / 'results' / 'xstreme'
MD_DIR = ROOT / 'results' / 'modisco'
# Which modisco h5 files to include. Default: single (all-celltype) only,
# i.e. exclude any '*_multitask*' h5. To use multitask instead, set e.g.
#   MODISCO_H5_SELECT = ['modisco_utr_shap_multitask']   (basenames without .h5)
# or set to None to include ALL h5 files found in MD_DIR.
MODISCO_H5_SELECT = 'single'   # 'single' | 'multitask'
def _modisco_h5_list():
    hs = sorted(MD_DIR.glob('*.h5')) if MD_DIR.exists() else []
    if MODISCO_H5_SELECT == 'multitask':
        return [h for h in hs if 'multitask' in h.stem]
    if MODISCO_H5_SELECT == 'single':
        return [h for h in hs if 'multitask' not in h.stem]
    sel = set(MODISCO_H5_SELECT)
    return [h for h in hs if h.stem in sel]
FIG_DIR = ROOT / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_rows', 200)
print('xstreme conds:', sorted(p.name for p in XS_DIR.glob('*') if p.is_dir()) if XS_DIR.exists() else 'none')
print('modisco h5   :', sorted(p.name for p in MD_DIR.glob('*.h5')) if MD_DIR.exists() else 'none')

# === FINALIZED SCOPE: te10 cutoff + 100nt window only (5p + 3p, hi + lo). ===
# Chosen because, considering both directions (hi/lo) and both ends (5'/3'),
# the te10 x 100nt conditions give the most consistent, interpretable motif
# themes (5'end GC-rich/CGG repressor + 3'end uORF/Kozak; pyrimidine activator).
TE_KEEP = {'te10'}
WIN_KEEP = {'5p100', '3p100'}

def keep_xstreme_cond(name):
    """XSTREME dir like 'te10_5p100_hi' -> keep only te10 + 5p100/3p100."""
    te = next((t for t in ('te10', 'te20') if t in name), None)
    win = next((w for w in ('5p100', '5p50', '3p100', '3p50', 'full_lenmatch') if w in name), None)
    return (te in TE_KEEP) and (win in WIN_KEEP)

print('FINALIZED scope -> te:', sorted(TE_KEEP), '| windows:', sorted(WIN_KEEP))


## 1. XSTREME - parse discovered/enriched motifs across conditions

In [ ]:
def parse_xstreme_tsv(path):
    """xstreme.tsv: tab-delimited, '#' comment lines. Returns DataFrame."""
    rows = []
    with open(path) as f:
        header = None
        for line in f:
            line = line.rstrip('\n')
            if not line or line.startswith('#'):
                continue
            parts = line.split('\t')
            if header is None:
                header = parts
                continue
            rows.append(parts)
    if header is None:
        return pd.DataFrame()
    df = pd.DataFrame(rows, columns=header[:len(rows[0])] if rows else header)
    return df

xs = {}
if XS_DIR.exists():
    for d in sorted(XS_DIR.glob('*')):
        if not (d.is_dir() and keep_xstreme_cond(d.name)):
            continue
        tsv = d / 'xstreme.tsv'
        if tsv.exists():
            df = parse_xstreme_tsv(tsv)
            df['condition'] = d.name
            xs[d.name] = df
if xs:
    xs_all = pd.concat(xs.values(), ignore_index=True)
    # 'detected_by': which program found the motif. XSTREME's SOURCE column holds
    # 'MEME'/'STREME' for de novo motifs, or a DB file path when the motif comes
    # from the known-motif database (reported via SEA). Map path -> 'SEA'.
    def _detector(src):
        s = str(src).strip()
        if s in ('MEME', 'STREME', 'DREME'):
            return s
        if '/' in s or s.endswith('.meme'):
            return 'SEA'          # known-DB motif (enrichment via SEA)
        return s or 'unknown'
    if 'SOURCE' in xs_all.columns:
        xs_all['detected_by'] = xs_all['SOURCE'].map(_detector)
        # also tag the known-DB name for SEA rows (basename of the path)
        xs_all['db_source'] = xs_all['SOURCE'].apply(
            lambda s: os.path.basename(str(s)) if ('/' in str(s) or str(s).endswith('.meme')) else '')
    # FINALIZED: exclude MEME-derived motifs from all analysis (keep STREME de novo
    # + SEA known-DB enrichments). Excluded by user request.
    if 'detected_by' in xs_all.columns:
        _n0 = len(xs_all)
        xs_all = xs_all[xs_all['detected_by'] != 'MEME'].reset_index(drop=True)
        print(f'  excluded MEME motifs: {_n0 - len(xs_all)} rows dropped, {len(xs_all)} kept')
    xs_all.to_csv(XS_DIR / "xs_all_finalized.tsv", header=True, sep="\t", index=False)
    print('total motif rows:', len(xs_all), '| columns:', list(xs_all.columns))
    display(xs_all.head(15))
else:
    print('No xstreme.tsv found yet. Run run_xstreme.sh first.')

In [ ]:
# Motif count per condition (how many significant motifs each comparison yielded)
if xs:
    counts = {k: len(v) for k, v in xs.items()}
    cs = pd.Series(counts).sort_index()
    print(cs)
    fig, ax = plt.subplots(figsize=(9, 4))
    cs.plot.bar(ax=ax, color='steelblue')
    ax.set_ylabel('# motifs (xstreme.tsv rows)')
    ax.set_title('XSTREME motif count per condition')
    plt.xticks(rotation=45, ha='right'); plt.tight_layout()
    fig.savefig(FIG_DIR / 'xstreme_motif_counts.png', dpi=150)
    plt.show()

## 4. TF-MoDISco - motifs, seqlet counts, TOMTOM matches
Read the modisco results h5: per-metacluster pattern count, seqlet support, and (if report run) TOMTOM matches.

In [ ]:
import h5py
import numpy as np

def _trim_consensus(pfm, contrib, frac=0.2):
    """Consensus string from a modisco pattern PFM (L,4), trimmed to the
    high-contribution core (|contrib| >= frac*max)."""
    score = np.abs(contrib).sum(axis=1)
    if score.max() <= 0:
        lo, hi = 0, len(pfm)
    else:
        keep = np.where(score >= frac * score.max())[0]
        lo, hi = keep.min(), keep.max() + 1
    letters = np.array(list('AGCT'))
    return ''.join(letters[pfm[lo:hi].argmax(axis=1)])

def modisco_pattern_consensus(h5path):
    """Map 'pos_patterns.pattern_N'/'neg_patterns.pattern_N' -> consensus string."""
    out = {}
    with h5py.File(h5path, 'r') as f:
        for mc in f.keys():
            grp = f[mc]
            if not hasattr(grp, 'keys'):
                continue
            for pat in grp.keys():
                p = grp[pat]
                if 'sequence' in p:
                    pfm = np.array(p['sequence'])
                    contrib = np.array(p['contrib_scores']) if 'contrib_scores' in p else pfm
                    out[f'{mc}.{pat}'] = _trim_consensus(pfm, contrib)
    return out

def summarize_modisco_h5(h5path):
    rows = []
    cons_map = modisco_pattern_consensus(h5path)
    with h5py.File(h5path, 'r') as f:
        for mc in f.keys():  # pos_patterns / neg_patterns
            grp = f[mc]
            if not hasattr(grp, 'keys'):
                continue
            for pat in grp.keys():
                p = grp[pat]
                n_seqlets = None
                if 'seqlets' in p and 'n_seqlets' in p['seqlets']:
                    n_seqlets = int(np.array(p['seqlets']['n_seqlets'])[0])
                elif 'seqlets' in p and 'sequence' in p['seqlets']:
                    n_seqlets = p['seqlets']['sequence'].shape[0]
                rows.append({'metacluster': mc, 'pattern': pat,
                             'consensus': cons_map.get(f'{mc}.{pat}', ''),
                             'n_seqlets': n_seqlets})
    return pd.DataFrame(rows)

if MD_DIR.exists():
    for h5 in _modisco_h5_list():
        print(f'=== {h5.name} ===')
        try:
            mdf = summarize_modisco_h5(h5)
            print(f'  metaclusters={mdf["metacluster"].nunique()}  patterns={len(mdf)}')
            display(mdf)
        except Exception as e:
            print('  error:', e)
else:
    print('Run run_modisco.sh first.')


In [ ]:
# TF-MoDISco TOMTOM matches.
# External tomtom path writes tomtom.tsv; tomtom-lite (-l, used here to avoid
# the padded-PPM crash) embeds matches in motifs.html instead. Parse whichever exists.
def load_modisco_matches(report_dir):
    report_dir = Path(report_dir)
    tt = report_dir / 'tomtom.tsv'
    if tt.exists():
        return pd.read_csv(tt, sep='\t'), 'tomtom.tsv'
    html = report_dir / 'motifs.html'
    if html.exists():
        tabs = pd.read_html(html)              # needs lxml
        df = tabs[0]
        # keep pattern + match/pval columns, drop embedded logo image columns
        keep = [c for c in df.columns if c in ('pattern','num_seqlets')
                or c.startswith('match') and not c.endswith('_logo')
                or c.startswith('pval')]
        return df[keep], 'motifs.html (tomtom-lite)'
    return None, None

if MD_DIR.exists():
    for rep in sorted(MD_DIR.glob('report_*')):
        m, src = load_modisco_matches(rep)
        if m is None:
            print(f'{rep.name}: no tomtom.tsv or motifs.html'); continue
        print(f'=== {rep.name} matches (from {src}, {len(m)} patterns) ===')
        display(m)
else:
    print('Run run_modisco.sh first.')


## 5. Unified motif table (STREME + SEA + TF-MoDISco)
One row per motif with tool-native scores. STREME/SEA carry `sites`/`evalue` (rank by `-log10(evalue)`); TF-MoDISco carries `n_seqlets`/`sum_contrib`. Blank cells = score not provided by that tool. `direction` up/down, `location` 5p/3p (STREME/SEA only), `subgroup` high/low (STREME/SEA only).


In [ ]:
# === Unified motif table + per-motif PWM (for R seqlogo). ===
# Table columns: tool, motif, consensus, sites, evalue, n_seqlets, sum_contrib,
#                direction, location. (MEME already excluded; no 'subgroup'.)
# STREME/SEA rows are sorted by evalue ascending WITHIN each (direction, location).
# PWM long: motif, tool, position, A, C, G, T  (probabilities, row-sum 1).
import re
# Known-motif MEME DB used for SEA enrichment (same as run_xstreme.sh MOTIF_DB).
# Point this to your MEME Suite motif_databases, or set the MOTIF_DB env var.
RBP_DB = os.environ.get('MOTIF_DB', 'motif_databases/RNA/Ray2013_rbp_Homo_sapiens.meme')

def parse_meme_pwms(path):
    """MEME minimal -> {(motif_id, alt_id): pwm (WxA)}."""
    out = {}
    if not Path(path).exists():
        return out
    lines = open(path).read().splitlines()
    i = 0
    while i < len(lines):
        if lines[i].startswith('MOTIF'):
            toks = lines[i].split()
            mid = toks[1] if len(toks) > 1 else ''
            alt = toks[2] if len(toks) > 2 else ''
            j = i + 1
            while j < len(lines) and not lines[j].startswith('letter-probability') \
                    and not lines[j].startswith('MOTIF'):
                j += 1
            rows = []
            if j < len(lines) and lines[j].startswith('letter-probability'):
                j += 1
                while j < len(lines) and lines[j].strip() and not lines[j].startswith('MOTIF'):
                    parts = lines[j].split()
                    if len(parts) >= 4:
                        try:
                            rows.append([float(x) for x in parts[:4]])
                        except ValueError:
                            break
                    j += 1
                if rows:
                    out[(mid, alt)] = np.array(rows)
            i = j
        else:
            i += 1
    return out

# SEA PWMs: Ray2013 DB keyed by ID (RNCMPT...)
_db = parse_meme_pwms(RBP_DB)
sea_pwm = {mid: pwm for (mid, alt), pwm in _db.items()}

# STREME PWMs: per-condition combined.meme keyed by (condition, STREME-N tag)
streme_pwm = {}
if XS_DIR.exists():
    for d in sorted(XS_DIR.glob('*')):
        if not (d.is_dir() and keep_xstreme_cond(d.name)):
            continue
        for (mid, alt), pwm in parse_meme_pwms(d / 'combined.meme').items():
            m = re.search(r'(STREME-\d+)', alt)
            if m:
                streme_pwm[(d.name, m.group(1))] = pwm

def _core_trim(pfm, contrib, frac=0.2):
    s = np.abs(contrib).sum(axis=1)
    if s.max() <= 0:
        return pfm
    keep = np.where(s >= frac * s.max())[0]
    return pfm[keep.min():keep.max() + 1]

rows, pwm_query, pwm_ref = [], [], []
def _add_pwm(target, tool, mid, motif, direction, location, pwm):
    pwm = np.asarray(pwm, float)
    pwm = pwm / np.clip(pwm.sum(1, keepdims=True), 1e-9, None)  # row-normalize
    for pos in range(pwm.shape[0]):
        a, c, g, t = pwm[pos]
        target.append({'tool': tool, 'id': mid, 'motif': motif, 'direction': direction,
                       'location': location, 'position': pos + 1,
                       'A': a, 'C': c, 'G': g, 'T': t})

# --- STREME + SEA from xs_all (te10_*p100, MEME excluded) ---
if 'xs_all' in dir() and len(xs_all):
    for _, r in xs_all.iterrows():
        loc = '5p' if '5p' in str(r['condition']) else ('3p' if '3p' in str(r['condition']) else '')
        dirn = 'up' if str(r['condition']).endswith('_hi') else ('down' if str(r['condition']).endswith('_lo') else '')
        motif_id = r.get('ALT_ID', '') if pd.notna(r.get('ALT_ID', '')) else ''
        rows.append({'tool': r['detected_by'], 'id': str(r.get('ID', '')), 'motif': motif_id,
                     'consensus': r.get('CONSENSUS', ''),
                     'sites': r.get('SITES', ''), 'evalue': r.get('EVALUE', ''),
                     'n_seqlets': '', 'sum_contrib': '',
                     'direction': dirn, 'location': loc})
        # attach PWM: SEA = reference (DB), STREME = query (learned) [+ matched reference]
        rid = str(r.get('ID', ''))
        if r['detected_by'] == 'SEA':
            pwm = sea_pwm.get(rid)
            if pwm is not None:
                _add_pwm(pwm_ref, 'SEA', rid, motif_id, dirn, loc, pwm)
        else:  # STREME de novo -> query PWM
            pwm = streme_pwm.get((str(r['condition']), str(motif_id)))
            if pwm is not None:
                _add_pwm(pwm_query, 'STREME', rid, motif_id, dirn, loc, pwm)
            # matched known reference (SIM_MOTIF -> Ray2013 RNCMPT) if any
            sim = str(r.get('SIM_MOTIF', ''))
            mm = re.match(r'(RNCMPT\d+)', sim)
            if 'Ray2013' in str(r.get('SIM_SOURCE', '')) and mm and mm.group(1) in sea_pwm:
                _add_pwm(pwm_ref, 'STREME', rid, motif_id, dirn, loc, sea_pwm[mm.group(1)])

# --- TF-MoDISco from h5 (core-trimmed PWM) ---
if MD_DIR.exists():
    for h5 in _modisco_h5_list():
        cons = modisco_pattern_consensus(h5)
        with h5py.File(h5, 'r') as f:
            for mc in f.keys():
                grp = f[mc]
                if not hasattr(grp, 'keys'):
                    continue
                dirn = 'up' if 'pos' in mc else ('down' if 'neg' in mc else '')
                for pat in grp.keys():
                    pobj = grp[pat]
                    if 'contrib_scores' not in pobj or 'sequence' not in pobj:
                        continue
                    cs = np.array(pobj['contrib_scores'])
                    pfm = np.array(pobj['sequence'])
                    # modisco h5 channels = dataset order A,G,C,T -> reorder to A,C,G,T
                    _R = [0, 2, 1, 3]; pfm = pfm[:, _R]; cs = cs[:, _R]
                    mid = f'{mc}.{pat}'
                    ns = int(np.array(pobj['seqlets']['n_seqlets']).ravel()[0]) \
                        if 'seqlets' in pobj and 'n_seqlets' in pobj['seqlets'] else ''
                    rows.append({'tool': 'TF-MoDISco', 'id': mid, 'motif': mid,
                                 'consensus': cons.get(mid, ''),
                                 'sites': '', 'evalue': '',
                                 'n_seqlets': ns, 'sum_contrib': float(np.abs(cs).sum()),
                                 'direction': dirn, 'location': ''})
                    _add_pwm(pwm_query, 'TF-MoDISco', mid, mid, dirn, '', _core_trim(pfm, cs))

COLS = ['tool', 'id', 'motif', 'consensus', 'sites', 'evalue', 'n_seqlets',
        'sum_contrib', 'direction', 'location']
motif_table = pd.DataFrame(rows, columns=COLS)

# STREME+SEA: sort by evalue asc within (direction, location); TF-MoDISco appended after.
ss = motif_table[motif_table.tool.isin(['STREME', 'SEA'])].copy()
ss['_ev'] = pd.to_numeric(ss['evalue'], errors='coerce')
ss = ss.sort_values(['direction', 'location', '_ev']).drop(columns='_ev')
md = motif_table[motif_table.tool == 'TF-MoDISco']
motif_table = pd.concat([ss, md], ignore_index=True)

motif_table.to_csv(ROOT / 'results' / 'unified_motif_table.tsv', sep='\t', index=False)
_PCOLS = ['tool', 'id', 'motif', 'direction', 'location', 'position', 'A', 'C', 'G', 'T']
pwm_query_df = pd.DataFrame(pwm_query, columns=_PCOLS)
pwm_ref_df = pd.DataFrame(pwm_ref, columns=_PCOLS)
pwm_query_df.to_csv(ROOT / 'results' / 'unified_motif_pwm_query.tsv', sep='\t', index=False)
pwm_ref_df.to_csv(ROOT / 'results' / 'unified_motif_pwm_reference.tsv', sep='\t', index=False)
print('unified_motif_table:', motif_table.shape, '| by tool:', dict(motif_table.tool.value_counts()))
print('query PWM:', pwm_query_df.shape, '| motifs:',
      pwm_query_df.groupby(['tool','id','direction','location']).ngroups,
      '(STREME de novo + TF-MoDISco)')
print('reference PWM:', pwm_ref_df.shape, '| motifs:',
      pwm_ref_df.groupby(['tool','id','direction','location']).ngroups,
      '(SEA DB + STREME-matched known)')
display(motif_table.head(20))


## 7. Motif visualization (top-5 by E-value, reference seqlogo)
Per (location, direction): top-5 STREME/SEA motifs by E-value. Left=5'p, right=3'p; UP block then DOWN block separated by a line. Each row: -log10(E-value) bar, reference seqlogo, motif name. `\u2014` = no reference PWM (novel STREME).


In [ ]:
# === Motif visualization: top-5 by E-value per (location, direction). ===
# Left subpanel = 5p, right = 3p. Within each: a SINGLE -log10(E-value) bar panel
# (all 10 motifs), then reference seqlogos (one row each) + names. UP block over
# DOWN block, separated by a gap/line.
import matplotlib.gridspec as gridspec
import logomaker

_rp = pd.read_csv(ROOT / 'results' / 'unified_motif_pwm_reference.tsv', sep='\t')

def _pwm_mat(df, key):
    s = df[(df.tool==key[0])&(df.id==key[1])&(df.direction==key[2])&(df.location==key[3])]
    if len(s) == 0:
        return None
    return s.sort_values('position')[['A','C','G','T']].reset_index(drop=True)

def _logo(ax, mat):
    if mat is None or len(mat) == 0:
        ax.text(0.5, 0.5, '\u2014', ha='center', va='center', fontsize=11, color='grey')
        ax.axis('off'); return
    info = logomaker.transform_matrix(mat, from_type='probability', to_type='information')
    logomaker.Logo(info, ax=ax, color_scheme='classic')
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(False)

_ss = motif_table[motif_table.tool.isin(['STREME', 'SEA'])].copy()
_ss['_ev'] = pd.to_numeric(_ss['evalue'], errors='coerce')
_ss['_nlp'] = -np.log10(_ss['_ev'].clip(lower=1e-300))
TOP = 5
_xmax = _ss['_nlp'].replace([np.inf], np.nan).max() * 1.05
GAP = 0.7  # vertical gap (in row units) between UP and DOWN blocks

fig = plt.figure(figsize=(13, 9))
outer = gridspec.GridSpec(1, 2, wspace=0.22)
_loc_title = {'5p': "5' UTR (cap-proximal)", '3p': "5' UTR (start-proximal)"}
for col, loc in enumerate(['5p', '3p']):
    # logo/name rows: TOP up + 1 gap + TOP down = 2*TOP+1 grid rows; +1 header
    NR = TOP*2 + 1
    inner = gridspec.GridSpecFromSubplotSpec(
        NR + 1, 3, subplot_spec=outer[col], wspace=0.1, hspace=0.25,
        width_ratios=[1.1, 2.2, 1.0])
    for c, lab in enumerate(['-log10 E', 'reference motif', 'name']):
        axh = fig.add_subplot(inner[0, c]); axh.axis('off')
        axh.text(0.5, 0.0, lab, ha='center', va='bottom', fontsize=9, fontweight='bold')
    fig.text(0.30 if col == 0 else 0.72, 0.93, _loc_title[loc],
             ha='center', fontsize=12, fontweight='bold')
    # single bar axes spanning all logo rows (grid rows 1..NR)
    axbar = fig.add_subplot(inner[1:NR + 1, 0])
    bar_y, bar_v, bar_c = [], [], []
    grid_row = 1
    for di, dr in enumerate(['up', 'down']):
        sub = _ss[(_ss.location==loc)&(_ss.direction==dr)].sort_values('_ev').head(TOP)
        # logo y-center for grid_row r maps to bar coordinate (NR - r + 0.5)
        for _, r in sub.iterrows():
            key = (r['tool'], r['id'], dr, loc)
            name = r['motif'] if pd.notna(r['motif']) and str(r['motif']).strip() else r['id']
            ycoord = NR - grid_row + 0.5
            bar_y.append(ycoord); bar_v.append(r['_nlp'])
            bar_c.append('#1f77b4' if dr == 'up' else '#d62728')
            _logo(fig.add_subplot(inner[grid_row, 1]), _pwm_mat(_rp, key))
            axn = fig.add_subplot(inner[grid_row, 2]); axn.axis('off')
            axn.text(0, 0.5, str(name), fontsize=11, va='center')
            grid_row += 1
        if di == 0:
            grid_row += 1  # skip a grid row for the gap/divider
    axbar.barh(bar_y, bar_v, color=bar_c, height=0.7)
    axbar.set_ylim(0, NR); axbar.set_xlim(0, _xmax)
    axbar.set_yticks([]); axbar.tick_params(labelsize=7)
    axbar.axhline(NR - (TOP + 0.5), color='k', lw=1.0)  # UP/DOWN divider
    for sp in ['top', 'right', 'left']: axbar.spines[sp].set_visible(False)
fig.suptitle('Top-5 motifs per (location, direction) by E-value  |  '
             'bar = -log10(E-value)  |  blue=UP, red=DOWN', fontsize=11, y=0.97)
fig.savefig(FIG_DIR / 'motif_visualization.png', dpi=140, bbox_inches='tight')
plt.show()
print('saved', FIG_DIR / 'motif_visualization.png')


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV

# Lasso(CV) sparse motif selection + held-out R2. FAST version.
# Old version was ~1h because it ran LassoCV().fit() (already a 5-fold CV over
# the alpha path) and THEN cross_val_score(LassoCV, cv=5) -> nested CV (LassoCV
# refit 5 more times). The held-out R2 is taken directly from LassoCV's own CV
# mse path (no second CV). Plus: fewer alphas, lower max_iter, float32, n_jobs.
def lasso_select(feat, label, n_jobs=4):
    df = feat.join(meta[['TE'] + COV], how='inner').dropna()
    y = df['TE'].values.astype(np.float32)
    Xcols = [c for c in feat.columns if c in df.columns] + COV
    X = StandardScaler().fit_transform(df[Xcols].values).astype(np.float32)
    las = LassoCV(cv=5, n_alphas=30, max_iter=2000, tol=1e-3,
                  selection='random', random_state=42, n_jobs=n_jobs).fit(X, y)
    # held-out R2 from LassoCV's own CV path (mse_path: (n_alphas, n_folds)),
    # converted to R2 at the selected alpha. No second cross_val_score.
    best = np.argmin(las.mse_path_.mean(axis=1))
    cv_mse = las.mse_path_[best].mean()
    cv_r2 = 1.0 - cv_mse / y.var()
    sel = pd.Series(las.coef_, index=Xcols)
    sel = sel[sel.abs() > 1e-6].sort_values(key=np.abs, ascending=False)
    print(f'=== {label} Lasso | alpha={las.alpha_:.4g}  CV R2(approx)={cv_r2:.4f}  '
          f'selected {len(sel)}/{len(Xcols)} ===')
    display(sel.head(25).to_frame('lasso_coef'))
    return las, sel

if labels:
    lasso_select(feat, labels[0])
